In [40]:
# Step 1: Goal
# Ingest exactly 3 PDFs, add production metadata, chunk, embed, and store in Chroma.

In [41]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [42]:
# Step 2: Load environment
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing. Add it to .env before embedding.")

In [ ]:
# Step 3: Config
DATA_DIR = Path("data")
target_count = 3
persist_directory = "chroma_store"
batch_size = 200

In [44]:
# Step 4: Select exactly 3 PDFs
pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
if len(pdf_files) < target_count:
    raise ValueError(f"Need at least {target_count} PDFs in {DATA_DIR.resolve()}, found {len(pdf_files)}")

target_pdfs = pdf_files[:target_count]
ingestion_run_id = f"run_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"

print("Ingestion run:", ingestion_run_id)
print("Selected PDFs:")
for p in target_pdfs:
    print("-", p)

Ingestion run: run_20260727T234039Z_8036d341
Selected PDFs:
- data\Atomic habits ( PDFDrive ).pdf
- data\attention.pdf
- data\BhagavadGita.pdf


In [45]:
# Step 5: Helper functions
def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


def load_with_metadata(pdf_path: Path, run_id: str):
    loader = PyMuPDFLoader(str(pdf_path))
    docs = loader.load()

    source_checksum = file_sha256(pdf_path)
    source_id = pdf_path.stem.lower().replace(" ", "_")

    for i, doc in enumerate(docs):
        doc_id = sha256(f"{pdf_path.resolve()}::{i}".encode()).hexdigest()
        doc.metadata.update(
            {
                "doc_id": doc_id,
                "source_id": source_id,
                "source_type": "pdf",
                "title": pdf_path.stem,
                "language": "en",
                "version": "v1",
                "checksum": source_checksum,
                "access_level": "internal",
                "pii_level": "none",
                "ingestion_run_id": run_id,
                "ingested_at": datetime.now(timezone.utc).isoformat(),
            }
        )
    return docs

In [46]:
# Step 6: Load all 3 PDFs
documents = []
for pdf_path in target_pdfs:
    loaded_docs = load_with_metadata(pdf_path, ingestion_run_id)
    documents.extend(loaded_docs)

print("Total loaded pages:", len(documents))

Total loaded pages: 1223


In [61]:
# Step 7: Preview one document metadata
documents[10].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'data\\Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits ( PDFDrive )',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 10,
 'doc_id': 'c5f78afec75f6701a4ae0c7b7cb27120ea35dd1cb03b95e9da5493e53f3ee852',
 'source_id': 'atomic_habits_(_pdfdrive_)',
 'source_type': 'pdf',
 'language': 'en',
 'version': 'v1',
 'checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'access_level': 'internal',
 'pii_level': 'none',
 'ingestion_run_id': 'run_20260727T234039Z_8036d341',
 'ingested_at': '2026-07-27T23:40:49.364712+00:00'}

In [60]:
# Step 7: Preview one document metadata
documents[1000].metadata

{'producer': 'Mac OS X 10.7.3 Quartz PDFContext',
 'creator': 'Pages',
 'creationdate': "D:20120508142201Z00'00'",
 'source': 'data\\BhagavadGita.pdf',
 'file_path': 'data\\BhagavadGita.pdf',
 'total_pages': 952,
 'format': 'PDF 1.3',
 'title': 'BhagavadGita',
 'author': 'me',
 'subject': '',
 'keywords': '',
 'moddate': "D:20120508142201Z00'00'",
 'trapped': '',
 'modDate': "D:20120508142201Z00'00'",
 'creationDate': "D:20120508142201Z00'00'",
 'page': 729,
 'doc_id': '7037db7fc53d9bbae67b67769597e7286ef2bed2479cb26d9ab4a5b26118635b',
 'source_id': 'bhagavadgita',
 'source_type': 'pdf',
 'language': 'en',
 'version': 'v1',
 'checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583',
 'access_level': 'internal',
 'pii_level': 'none',
 'ingestion_run_id': 'run_20260727T234039Z_8036d341',
 'ingested_at': '2026-07-27T23:40:52.801483+00:00'}

In [66]:
# Step 7: Preview one document metadata
documents[1200].metadata

{'producer': 'Mac OS X 10.7.3 Quartz PDFContext',
 'creator': 'Pages',
 'creationdate': "D:20120508142201Z00'00'",
 'source': 'data\\BhagavadGita.pdf',
 'file_path': 'data\\BhagavadGita.pdf',
 'total_pages': 952,
 'format': 'PDF 1.3',
 'title': 'BhagavadGita',
 'author': 'me',
 'subject': '',
 'keywords': '',
 'moddate': "D:20120508142201Z00'00'",
 'trapped': '',
 'modDate': "D:20120508142201Z00'00'",
 'creationDate': "D:20120508142201Z00'00'",
 'page': 929,
 'doc_id': '70b3c130ab8df0d9d2b298d8a0751b31331b4ff50c8fb33a151e351b53ea19d4',
 'source_id': 'bhagavadgita',
 'source_type': 'pdf',
 'language': 'en',
 'version': 'v1',
 'checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583',
 'access_level': 'internal',
 'pii_level': 'none',
 'ingestion_run_id': 'run_20260727T234039Z_8036d341',
 'ingested_at': '2026-07-27T23:40:52.941110+00:00'}

In [67]:
# Simple check: unique source_id and title
unique_source_ids = sorted({d.metadata.get("source_id", "unknown") for d in documents})
unique_titles = sorted({d.metadata.get("title", "unknown") for d in documents})

print("Unique source_id:", unique_source_ids)
print("Unique title:", unique_titles)

Unique source_id: ['atomic_habits_(_pdfdrive_)', 'attention', 'bhagavadgita']
Unique title: ['Atomic habits ( PDFDrive )', 'BhagavadGita', 'attention']


In [68]:
# Step 8: Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = splitter.split_documents(documents)

In [69]:
# Step 9: Add chunk IDs and minimal chunk metadata
for chunk_idx, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = sha256(
        f"{chunk.metadata.get('doc_id')}::{chunk_idx}".encode()
    ).hexdigest()
    chunk.metadata["chunk_index"] = chunk_idx

print("Total chunks:", len(chunks))

Total chunks: 2837


In [72]:
chunks[3].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'data\\Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits ( PDFDrive )',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 5,
 'doc_id': '5ae0120766b5296b98841c19671c2e6042ef5e711658bdf8cac03a1476e98fbd',
 'source_id': 'atomic_habits_(_pdfdrive_)',
 'source_type': 'pdf',
 'language': 'en',
 'version': 'v1',
 'checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'access_level': 'internal',
 'pii_level': 'none',
 'ingestion_run_id': 'run_20260727T234039Z_8036d341',
 'ingested_at': '2026-07-27T23:40:49.363723+00:00',
 'chunk_id': 'd0a01bf57c655fa3a74eb14aa3363ffc4548383133b946df1f0

In [73]:
# Step 10: Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# Step 11: Build collection names per PDF source (single vector store)
def safe_collection_suffix(value: str) -> str:
    cleaned = "".join(ch if (ch.isalnum() or ch in "._-") else "_" for ch in value)
    cleaned = cleaned.strip("._-")
    return cleaned or "unknown"


Persist directory: chroma_store
Collections to create:
- atomic_habits_(_pdfdrive_) -> atomic_habits___pdfdrive
- attention -> attention
- bhagavadgita -> bhagavadgita


In [79]:


source_ids = sorted({d.metadata.get("source_id", "unknown") for d in chunks})
source_ids

['atomic_habits_(_pdfdrive_)', 'attention', 'bhagavadgita']

In [80]:

collection_names = {
    source_id: safe_collection_suffix(source_id)
    for source_id in source_ids
}

collection_names

{'atomic_habits_(_pdfdrive_)': 'atomic_habits___pdfdrive',
 'attention': 'attention',
 'bhagavadgita': 'bhagavadgita'}

In [82]:

print("Persist directory:", persist_directory)



Persist directory: chroma_store


In [83]:
print("Collections to create:")

for source_id, name in collection_names.items():
    print(f"- {source_id} -> {name}")

Collections to create:
- atomic_habits_(_pdfdrive_) -> atomic_habits___pdfdrive
- attention -> attention
- bhagavadgita -> bhagavadgita


In [84]:
# Step 12: Ingest into multiple collections in one vector store
collection_stores = {}

for source_id, collection_name in collection_names.items():
    store = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=persist_directory,
    )
    collection_stores[source_id] = store

In [85]:
collection_stores

{'atomic_habits_(_pdfdrive_)': <langchain_chroma.vectorstores.Chroma at 0x1e789955eb0>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1e789956330>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1e78347adb0>}

In [98]:
len(chunks)

2837

In [ ]:
for source_id, store in collection_stores.items():
    print("adding documents to collection:", source_id)
    print("collection name:", store._collection.name)

    

    break
    # source_chunks = []
    # for d in chunks:
    #     if d.metadata.get("source_id") == source_id:
    #         source_chunks.append(d)
    #     print(d)
       
    # source_ids = []
    # for d in source_chunks:
    #     source_ids.append(d.metadata["chunk_id"])

    # for start in range(0, len(source_chunks), batch_size):
    #     end = start + batch_size
    #     store.add_documents(
    #         documents=source_chunks[start:end],
    #         ids=source_ids[start:end],
    #     )

# print("Collections ingested:", len(collection_stores))

adding documents to collection: atomic_habits_(_pdfdrive_)
collection name: atomic_habits___pdfdrive


In [57]:
# Step 13: Ingestion summary by collection
for source_id, store in collection_stores.items():
    collection_name = collection_names[source_id]
    count = store._collection.count()
    print(f"{collection_name}: {count} chunks")

gita_vector_db_atomic_habits___pdfdrive: 632 chunks
gita_vector_db_attention: 49 chunks
gita_vector_db_bhagavadgita: 2156 chunks


In [58]:
# Step 14: Total chunks across all collections
total_count = sum(store._collection.count() for store in collection_stores.values())
print("Total chunks across collections:", total_count)

Total chunks across collections: 2837
